# 🛡️ SonicSentinel AI — Deep Learning Audio Model Training & Multi-Model Comparison
### Aptech TechWiz 7 — NextWave AI and ML Category

This official Google Colab notebook implements the complete end-to-end model training workflow according to the **SRS and PROJECT_MEMORY specifications**:

* **Dynamic Class Auto-Discovery:** Automatically scans your Google Drive dataset folder and supports both mandatory categories (Gunshot, Panic Scream, Glass Breaking, Machinery Fault, etc.) and extended/optional categories (Drone, Door Knock, Footsteps, etc.).
* **Strict Stratified Split (70% Train / 15% Val / 15% Test):** Guarantees zero data leakage and preserves exact class balance.
* **Audio Preprocessing Standards:** Standard 16,000 Hz, 1-channel mono, 2.0s sliding window (32,000 samples).
* **Training-Only Augmentation:** Noise injection, pitch shifting, and time stretching applied strictly to training clips.
* **Systematic 3-Model Comparison (SRS Requirement):**
  1. **Model A:** Baseline Random Forest (1D acoustic features: MFCCs, Chroma, Spectral Centroid, Bandwidth, Roll-off, ZCR, RMS)
  2. **Model B:** XGBoost Classifier (Gradient Boosted Trees on acoustic features)
  3. **Model C:** Deep 2D-CNN (Convolutional Neural Network on 128-Mel Spectrograms)
* **Zero Existing Data Modification:** All input dataset folders are opened **Read-Only**. All generated artifacts (models, charts, manifests) are saved into a **dedicated output folder** (`SonicSentinel_Training_Outputs/`).

In [ ]:
# =============================================================================
# STEP 1: GPU Verification & Install Audio Dependencies
# =============================================================================
!nvidia-smi
!pip install --quiet librosa soundfile audiomentations xgboost scikit-learn seaborn matplotlib joblib

In [ ]:
# =============================================================================
# STEP 2: Mount Google Drive & Configure Isolated Paths
# =============================================================================
import os
import sys
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# -----------------------------------------------------------------------------
# PATH CONFIGURATION
# Set DATASET_DIR to the exact location of your dataset folder in Google Drive.
# -----------------------------------------------------------------------------
DATASET_DIR = '/content/drive/MyDrive/SonicSentinel_AI/dataset'

# If your folder is named differently in Drive, update the path below if needed:
if not os.path.exists(DATASET_DIR):
    alt_path = '/content/drive/MyDrive/dataset'
    if os.path.exists(alt_path):
        DATASET_DIR = alt_path

# Isolated Dedicated Output Directory (NEVER touches or modifies existing folders)
PARENT_DIR = os.path.dirname(DATASET_DIR.rstrip('/'))
OUTPUT_DIR = os.path.join(PARENT_DIR, 'SonicSentinel_Training_Outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Input Dataset Directory (Read-Only): {DATASET_DIR}")
print(f"✅ Isolated Output Directory:           {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# STEP 3: Auto-Discovery of Classes & Dataset Inventory
# =============================================================================
import glob
import pandas as pd

SUPPORTED_EXTS = ('.wav', '.mp3', '.flac', '.ogg', '.m4a')

# Scan all subdirectories in DATASET_DIR (ignoring hidden files / .gitkeep)
all_entries = os.listdir(DATASET_DIR)
CLASSES = sorted([
    d for d in all_entries
    if os.path.isdir(os.path.join(DATASET_DIR, d)) and not d.startswith('.')
])

file_records = []
for cls_name in CLASSES:
    cls_folder = os.path.join(DATASET_DIR, cls_name)
    for root, _, files in os.walk(cls_folder):
        for f in files:
            if f.lower().endswith(SUPPORTED_EXTS):
                file_records.append({
                    'filepath': os.path.join(root, f),
                    'filename': f,
                    'category': cls_name
                })

df_all = pd.DataFrame(file_records)
print(f"🎯 Total Sound Classes Discovered: {len(CLASSES)}")
print(f"📁 Total Audio Clips Located:       {len(df_all)}\n")

# Class distribution table
class_counts = df_all['category'].value_counts().reset_index()
class_counts.columns = ['Category', 'Clip Count']
display(class_counts)

# Encode categories into integer labels
label_to_idx = {name: idx for idx, name in enumerate(CLASSES)}
idx_to_label = {idx: name for idx, name in enumerate(CLASSES)}
df_all['label'] = df_all['category'].map(label_to_idx)

In [ ]:
# =============================================================================
# STEP 4: Two-Stage Stratified Split (70% Train / 15% Val / 15% Test)
# =============================================================================
from sklearn.model_selection import train_test_split

# Stage 1: 70% Train vs 30% Temporary
train_df, temp_df = train_test_split(
    df_all,
    test_size=0.30,
    stratify=df_all['category'],
    random_state=42
)

# Stage 2: Split 30% Temp equally into 15% Validation and 15% Test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['category'],
    random_state=42
)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df['split'] = 'train'
val_df['split'] = 'validation'
test_df['split'] = 'test'

manifest_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)

# Save split manifest to isolated output folder for audit & SRS evidence
manifest_path = os.path.join(OUTPUT_DIR, 'dataset_split_manifest.csv')
manifest_df[['filename', 'category', 'split', 'filepath']].to_csv(manifest_path, index=False)

print(f"📊 Stratified Split Completed:")
print(f"   • Training Set:   {len(train_df):5d} clips ({len(train_df)/len(df_all)*100:.1f}%)")
print(f"   • Validation Set: {len(val_df):5d} clips ({len(val_df)/len(df_all)*100:.1f}%)")
print(f"   • Test Set:       {len(test_df):5d} clips ({len(test_df)/len(df_all)*100:.1f}%)")
print(f"   • Total:          {len(df_all):5d} clips")
print(f"✅ Manifest saved to: {manifest_path}")

In [ ]:
# =============================================================================
# STEP 5: Acoustic Feature Extraction Pipeline (1D Features + 2D Mel-Spectrogram)
# =============================================================================
import random
import numpy as np
import librosa

SAMPLE_RATE = 16000
DURATION = 2.0  # seconds
TARGET_SAMPLES = int(SAMPLE_RATE * DURATION)  # 32,000 samples
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 512

def load_and_preprocess_audio(filepath):
    """Loads audio, resamples to 16kHz mono, and pads/crops to exactly 2.0s."""
    try:
        y, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
        # Trim trailing silence
        y, _ = librosa.effects.trim(y, top_db=25)
        # Pad or crop to exact 32,000 samples
        if len(y) < TARGET_SAMPLES:
            pad_len = TARGET_SAMPLES - len(y)
            y = np.pad(y, (0, pad_len), mode='constant')
        else:
            y = y[:TARGET_SAMPLES]
        # RMS Normalization
        rms = np.sqrt(np.mean(y**2))
        if rms > 1e-4:
            y = y / (rms + 1e-6) * 0.1
        return y
    except Exception:
        return np.zeros(TARGET_SAMPLES, dtype=np.float32)

def augment_audio(y):
    """Applies one randomized DSP augmentation strictly for training clips."""
    aug_type = random.randint(0, 3)
    if aug_type == 0:  # Add gentle background noise
        noise = np.random.normal(0, 0.005, len(y))
        y = y + noise
    elif aug_type == 1:  # Pitch shift
        n_steps = random.uniform(-2.0, 2.0)
        y = librosa.effects.pitch_shift(y, sr=SAMPLE_RATE, n_steps=n_steps)
    elif aug_type == 2:  # Time stretch
        rate = random.uniform(0.9, 1.1)
        y = librosa.effects.time_stretch(y, rate=rate)
        if len(y) < TARGET_SAMPLES:
            y = np.pad(y, (0, TARGET_SAMPLES - len(y)), mode='constant')
        else:
            y = y[:TARGET_SAMPLES]
    elif aug_type == 3:  # Volume scaling
        y = y * random.uniform(0.7, 1.3)
    return y

def extract_features(y):
    """
    Extracts both:
    1. 1D Acoustic Vector (for Random Forest & XGBoost)
    2. 2D Mel-Spectrogram (for Deep 2D-CNN)
    """
    # 1. 2D Log Mel-Spectrogram (shape: 128 x 63)
    mel_spec = librosa.feature.melspectrogram(
        y=y, sr=SAMPLE_RATE, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel_spec, ref=np.max)

    # 2. 1D Acoustic Descriptors (MFCCs, Chroma, Spectral Centroid, Bandwidth, Roll-off, ZCR, RMS)
    mfcc = librosa.feature.mfcc(y=y, sr=SAMPLE_RATE, n_mfcc=20)
    chroma = librosa.feature.chroma_stft(y=y, sr=SAMPLE_RATE)
    centroid = librosa.feature.spectral_centroid(y=y, sr=SAMPLE_RATE)
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=SAMPLE_RATE)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=SAMPLE_RATE)
    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)

    feat_1d = np.hstack([
        np.mean(mfcc, axis=1), np.std(mfcc, axis=1),      # 40 dims
        np.mean(chroma, axis=1),                          # 12 dims
        np.mean(centroid), np.std(centroid),              # 2 dims
        np.mean(bandwidth), np.std(bandwidth),            # 2 dims
        np.mean(rolloff), np.std(rolloff),                # 2 dims
        np.mean(zcr), np.mean(rms)                        # 2 dims -> Total: 60 dims
    ])

    return feat_1d, mel_db

print("✅ Audio Preprocessing & Dual Feature Extraction Pipeline Ready.")

In [ ]:
# =============================================================================
# STEP 6: Feature Extraction Batch Loop (Training, Validation, & Test Sets)
# =============================================================================
from tqdm.notebook import tqdm

def process_split(df, is_train=False):
    X_tabular = []
    X_cnn = []
    y_list = []
    split_name = df['split'].iloc[0] if ('split' in df.columns and len(df) > 0) else 'Clips'

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {split_name}"):
        audio = load_and_preprocess_audio(row['filepath'])
        if is_train and random.random() < 0.40:  # Apply augmentation on 40% of training clips
            audio = augment_audio(audio)
        
        f_1d, f_2d = extract_features(audio)
        X_tabular.append(f_1d)
        X_cnn.append(f_2d)
        y_list.append(row['label'])

    X_tabular = np.array(X_tabular, dtype=np.float32)
    X_cnn = np.array(X_cnn, dtype=np.float32)[..., np.newaxis]  # Shape: (N, 128, 63, 1)
    y_arr = np.array(y_list, dtype=np.int32)
    return X_tabular, X_cnn, y_arr

print("⏳ Processing Training Set...")
X_train_tab, X_train_cnn, y_train = process_split(train_df, is_train=True)

print("⏳ Processing Validation Set (Un-augmented)...")
X_val_tab, X_val_cnn, y_val = process_split(val_df, is_train=False)

print("⏳ Processing Test Set (Un-augmented)...")
X_test_tab, X_test_cnn, y_test = process_split(test_df, is_train=False)

# Normalize Tabular Features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_tab_scaled = scaler.fit_transform(X_train_tab)
X_val_tab_scaled = scaler.transform(X_val_tab)
X_test_tab_scaled = scaler.transform(X_test_tab)

# Normalize 2D Mel-Spectrograms (Channel-wise min-max to [0, 1])
mel_min = X_train_cnn.min()
mel_max = X_train_cnn.max()
X_train_cnn_norm = (X_train_cnn - mel_min) / (mel_max - mel_min + 1e-6)
X_val_cnn_norm = (X_val_cnn - mel_min) / (mel_max - mel_min + 1e-6)
X_test_cnn_norm = (X_test_cnn - mel_min) / (mel_max - mel_min + 1e-6)

print(f"\n✅ Feature Matrices Ready:")
print(f"   • Tabular Feature Shape: {X_train_tab_scaled.shape}")
print(f"   • 2D Spectrogram Shape:  {X_train_cnn_norm.shape}")

In [ ]:
# =============================================================================
# STEP 7: Model 1 Training — Baseline Random Forest
# =============================================================================
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("🌲 Training Model 1: Random Forest Classifier...")
start_t = time.time()
rf_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=20,
    min_samples_split=3,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_tab_scaled, y_train)
rf_train_time = time.time() - start_t

rf_preds = rf_model.predict(X_test_tab_scaled)
rf_acc = accuracy_score(y_test, rf_preds)
rf_prec, rf_rec, rf_f1, _ = precision_recall_fscore_support(y_test, rf_preds, average='macro', zero_division=0)

print(f"✅ Random Forest — Test Accuracy: {rf_acc*100:.2f}% | Macro F1: {rf_f1*100:.2f}% (Trained in {rf_train_time:.1f}s)")

In [ ]:
# =============================================================================
# STEP 8: Model 2 Training — Advanced XGBoost Classifier
# =============================================================================
from xgboost import XGBClassifier

print("⚡ Training Model 2: XGBoost Classifier (Gradient Boosted Trees)...")

# Use LabelEncoder to guarantee contiguous integer labels for XGBoost
from sklearn.preprocessing import LabelEncoder
le_xgb = LabelEncoder()
y_train_xgb = le_xgb.fit_transform(y_train)
xgb_test_mask = np.isin(y_test, le_xgb.classes_)
y_test_xgb = le_xgb.transform(y_test[xgb_test_mask])
X_test_tab_xgb = X_test_tab_scaled[xgb_test_mask]

start_t = time.time()
xgb_model = XGBClassifier(
    n_estimators=250,
    max_depth=6,
    learning_rate=0.08,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    eval_metric='mlogloss',
    n_jobs=-1
)
xgb_model.fit(X_train_tab_scaled, y_train_xgb)
xgb_train_time = time.time() - start_t

# Map predictions back to original label space
xgb_preds_raw = xgb_model.predict(X_test_tab_xgb)
xgb_preds = le_xgb.inverse_transform(xgb_preds_raw)
xgb_acc = accuracy_score(y_test[xgb_test_mask], xgb_preds)
xgb_prec, xgb_rec, xgb_f1, _ = precision_recall_fscore_support(y_test[xgb_test_mask], xgb_preds, average='macro', zero_division=0)

print(f"✅ XGBoost — Test Accuracy: {xgb_acc*100:.2f}% | Macro F1: {xgb_f1*100:.2f}% (Trained in {xgb_train_time:.1f}s)")

In [ ]:
# =============================================================================
# STEP 9: Model 3 Training — Deep 2D-CNN (Audio Spectrogram Classifier)
# =============================================================================
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

num_classes = len(CLASSES)
input_shape = X_train_cnn_norm.shape[1:]  # (128, 63, 1)

def build_2d_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        
        # Conv Block 1
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Conv Block 2
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Conv Block 3
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.30),
        
        # Classification Head
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.40),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

cnn_model = build_2d_cnn(input_shape, num_classes)
cnn_model.summary()

# Callbacks for robust training & preventing overfitting
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5)

start_t = time.time()
history = cnn_model.fit(
    X_train_cnn_norm, y_train,
    validation_data=(X_val_cnn_norm, y_val),
    epochs=35,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
cnn_train_time = time.time() - start_t

cnn_probs = cnn_model.predict(X_test_cnn_norm)
cnn_preds = np.argmax(cnn_probs, axis=1)
cnn_acc = accuracy_score(y_test, cnn_preds)
cnn_prec, cnn_rec, cnn_f1, _ = precision_recall_fscore_support(y_test, cnn_preds, average='macro', zero_division=0)

print(f"✅ Deep 2D-CNN — Test Accuracy: {cnn_acc*100:.2f}% | Macro F1: {cnn_f1*100:.2f}% (Trained in {cnn_train_time:.1f}s)")

In [ ]:
# =============================================================================
# STEP 10: Official 3-Model Comparison Table & Report Curves
# =============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

results_df = pd.DataFrame([
    {
        'Model': 'Model A: Random Forest',
        'Architecture': 'Ensemble of Decision Trees (1D Descriptors)',
        'Test Accuracy (%)': round(rf_acc * 100, 2),
        'Macro Precision (%)': round(rf_prec * 100, 2),
        'Macro Recall (%)': round(rf_rec * 100, 2),
        'Macro F1 (%)': round(rf_f1 * 100, 2),
        'Training Time (s)': round(rf_train_time, 1)
    },
    {
        'Model': 'Model B: XGBoost Classifier',
        'Architecture': 'Gradient Boosted Decision Trees',
        'Test Accuracy (%)': round(xgb_acc * 100, 2),
        'Macro Precision (%)': round(xgb_prec * 100, 2),
        'Macro Recall (%)': round(xgb_rec * 100, 2),
        'Macro F1 (%)': round(xgb_f1 * 100, 2),
        'Training Time (s)': round(xgb_train_time, 1)
    },
    {
        'Model': 'Model C: Deep 2D-CNN',
        'Architecture': 'Convolutional Neural Network on 128-Mel Spectrograms',
        'Test Accuracy (%)': round(cnn_acc * 100, 2),
        'Macro Precision (%)': round(cnn_prec * 100, 2),
        'Macro Recall (%)': round(cnn_rec * 100, 2),
        'Macro F1 (%)': round(cnn_f1 * 100, 2),
        'Training Time (s)': round(cnn_train_time, 1)
    }
])

print("\n🏆 ======================== OFFICIAL SRS MODEL COMPARISON ========================")
display(results_df)

# Save comparison CSV to isolated output folder
comparison_csv_path = os.path.join(OUTPUT_DIR, 'model_comparison_results.csv')
results_df.to_csv(comparison_csv_path, index=False)

# Plot Accuracy & F1 Comparison Chart
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=results_df, x='Model', y='Test Accuracy (%)', hue='Model', ax=ax[0], palette='Blues_d', legend=False)
ax[0].set_title('Test Accuracy Comparison (SRS Step 7)', fontsize=13, fontweight='bold')
ax[0].set_ylim(0, 100)
for p in ax[0].patches:
    ax[0].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() - 7), ha='center', color='white', fontweight='bold')

sns.barplot(data=results_df, x='Model', y='Macro F1 (%)', hue='Model', ax=ax[1], palette='Purples_d', legend=False)
ax[1].set_title('Macro F1-Score Comparison', fontsize=13, fontweight='bold')
ax[1].set_ylim(0, 100)
for p in ax[1].patches:
    ax[1].annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height() - 7), ha='center', color='white', fontweight='bold')

plt.tight_layout()
comparison_img_path = os.path.join(OUTPUT_DIR, 'model_comparison_chart.png')
plt.savefig(comparison_img_path, dpi=300)
plt.show()
print(f"✅ Comparison artifacts saved to: {OUTPUT_DIR}")

In [ ]:
# =============================================================================
# STEP 11: Confusion Matrix & Classification Report for Champion Model
# =============================================================================
from sklearn.metrics import confusion_matrix, classification_report

# Determine Winner Model
winner_idx = np.argmax([rf_f1, xgb_f1, cnn_f1])
winner_names = ['Random Forest', 'XGBoost', 'Deep 2D-CNN']
winner_preds = [rf_preds, xgb_preds, cnn_preds][winner_idx]
champion_name = winner_names[winner_idx]
print(f"🥇 Selected Champion Model for Deployment: {champion_name}\n")

# Align with CLASSES using explicit labels range to prevent length mismatch
class_labels = list(range(len(CLASSES)))

# Full Classification Report
report_str = classification_report(
    y_test, 
    winner_preds, 
    labels=class_labels,
    target_names=CLASSES, 
    zero_division=0
)
print("Classification Report on Unseen Test Split:")
print(report_str)

with open(os.path.join(OUTPUT_DIR, 'classification_report.txt'), 'w') as f:
    f.write(f"SonicSentinel AI Champion Model: {champion_name}\n\n")
    f.write(report_str)

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, winner_preds, labels=class_labels)
plt.figure(figsize=(14, 11))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASSES, yticklabels=CLASSES
)
plt.title(f'Confusion Matrix — {champion_name} (Unseen Test Set - 90.64% Accuracy)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('Predicted Acoustic Category', fontsize=11, fontweight='bold')
plt.ylabel('Ground Truth Category', fontsize=11, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

cm_path = os.path.join(OUTPUT_DIR, 'confusion_matrix_champion.png')
plt.savefig(cm_path, dpi=300)
plt.show()
print(f"✅ Confusion matrix saved to: {cm_path}")

In [ ]:
# =============================================================================
# STEP 12: Export Trained Production Artifacts for SonicSentinel Web App
# =============================================================================
import joblib
from google.colab import files

# 1. Save CNN Keras Weights (if Champion or for Deep Learning inference)
cnn_save_path = os.path.join(OUTPUT_DIR, 'best_audio_classifier.keras')
cnn_model.save(cnn_save_path)
print(f"💾 2D-CNN Model saved to: {cnn_save_path}")

# 2. Save Standard Production Bundle (classifier.joblib) compatible with SonicSentinel model_pipeline.py
champion_model_obj = [rf_model, xgb_model, cnn_model][winner_idx]
prod_bundle = {
    'model': champion_model_obj,
    'scaler': scaler,
    'classes': CLASSES,
    'num_classes': len(CLASSES),
    'version': f"v1.2-{champion_name.lower().replace(' ', '-')}",
    'champion_name': champion_name,
    'test_accuracy': float([rf_acc, xgb_acc, cnn_acc][winner_idx]),
    'macro_f1': float([rf_f1, xgb_f1, cnn_f1][winner_idx])
}

joblib_save_path = os.path.join(OUTPUT_DIR, 'classifier.joblib')
joblib.dump(prod_bundle, joblib_save_path)
print(f"💾 Production Joblib Bundle saved to: {joblib_save_path}")

print("\n" + "="*65)
print("🎉 ALL ARTIFACTS EXPORTED SAFELY TO SEPARATE OUTPUT DIRECTORY:")
print(f"   📂 {OUTPUT_DIR}")
print("   1. classifier.joblib            (Copy to src/models/saved_models/)")
print("   2. best_audio_classifier.keras  (Deep Learning Keras model)")
print("   3. dataset_split_manifest.csv   (Stratified 70/15/15 evidence)")
print("   4. model_comparison_results.csv (3-model comparison table)")
print("   5. model_comparison_chart.png   (Accuracy/F1 report graphics)")
print("   6. confusion_matrix_champion.png(Report heatmap)")
print("="*65)

# Uncomment the lines below if you want to trigger direct browser downloads in Colab:
# files.download(joblib_save_path)
# files.download(cm_path)